# Import libraries 

In [3]:
!pip install mlxtend
import pandas as pd
from mlxtend.frequent_patterns import apriori,association_rules
from mlxtend.preprocessing import TransactionEncoder
!pip install mlxtend

In [4]:
titanic = pd.read_csv(r"C:\Users\munes\Dropbox\PC\Documents\assignments\assignments\Titanic (1).csv")
titanic

,Class,Gender,Age,Survived
0,3rd,Male,Child,No
1,3rd,Male,Child,No
2,3rd,Male,Child,No
3,3rd,Male,Child,No
4,3rd,Male,Child,No
...,...,...,...,...
2196,Crew,Female,Adult,Yes
2197,Crew,Female,Adult,Yes
2198,Crew,Female,Adult,Yes
2199,Crew,Female,Adult,Yes


In [5]:
unique_classes = titanic['Class'].unique()
print(unique_classes)
class_counts = titanic.groupby('Class').size()
print(class_counts)
titanic.drop_duplicates(subset=['Class'])
'''The subset=['Class'] argument ensures that duplicates are checked only based on the 'Class' column.
Only the first occurrence of each unique class is kept, while duplicate occurrences are removed.
Other columns are not considered when identifying duplicates.'''

['3rd' '1st' '2nd' 'Crew']
Class
1st     325
2nd     285
3rd     706
Crew    885
dtype: int64


"The subset=['Class'] argument ensures that duplicates are checked only based on the 'Class' column.\nOnly the first occurrence of each unique class is kept, while duplicate occurrences are removed.\nOther columns are not considered when identifying duplicates."

In [6]:
class_summary = pd.crosstab(titanic['Class'], titanic['Survived'])
print(class_summary)
'''Best Approach for Unique Classes:
If you only need unique class names → Use .unique()
If you want distinct rows based on class → Use .drop_duplicates(subset=['Class'])
If you need to count occurrences → Use .groupby()
If you need a class-wise summary → Use pd.crosstab()'''

Survived   No  Yes
Class             
1st       122  203
2nd       167  118
3rd       528  178
Crew      673  212


"Best Approach for Unique Classes:\nIf you only need unique class names → Use .unique()\nIf you want distinct rows based on class → Use .drop_duplicates(subset=['Class'])\nIf you need to count occurrences → Use .groupby()\nIf you need a class-wise summary → Use pd.crosstab()"

# Pre-Processing
As the data is not in transaction formation 
We are using transaction Encoder

In [8]:
df=pd.get_dummies(titanic)
df.head()

,Class_1st,Class_2nd,Class_3rd,Class_Crew,Gender_Female,Gender_Male,Age_Adult,Age_Child,Survived_No,Survived_Yes
0,False,False,True,False,False,True,False,True,True,False
1,False,False,True,False,False,True,False,True,True,False
2,False,False,True,False,False,True,False,True,True,False
3,False,False,True,False,False,True,False,True,True,False
4,False,False,True,False,False,True,False,True,True,False


# Apriori Algorithm 

In [10]:
frequent_itemsets = apriori(df, min_support=0.15, use_colnames=True)
frequent_itemsets

,support,itemsets
0,0.320763,(Class_3rd)
1,0.402090,(Class_Crew)
2,0.213539,(Gender_Female)
3,0.786461,(Gender_Male)
4,0.950477,(Age_Adult)
5,0.676965,(Survived_No)
6,0.323035,(Survived_Yes)
7,0.231713,"(Class_3rd, Gender_Male)"
8,0.284871,"(Class_3rd, Age_Adult)"
9,0.239891,"(Class_3rd, Survived_No)"


In [11]:
pd.set_option('display.max_rows',None)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
rules
rules.sort_values('lift',ascending = False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
8,(Gender_Female),(Survived_Yes),0.213539,0.323035,0.156293,0.731915,2.265745,1.0,0.087312,2.525187,0.710327,0.410992,0.603990,0.607870
9,(Survived_Yes),(Gender_Female),0.323035,0.213539,0.156293,0.483826,2.265745,1.0,0.087312,1.523634,0.825219,0.410992,0.343674,0.607870
60,"(Class_Crew, Survived_No)","(Age_Adult, Gender_Male)",0.305770,0.757383,0.304407,0.995542,1.314450,1.0,0.072822,54.427079,0.344592,0.401198,0.981627,0.698731
61,"(Age_Adult, Gender_Male)","(Class_Crew, Survived_No)",0.757383,0.305770,0.304407,0.401920,1.314450,1.0,0.072822,1.160764,0.986022,0.401198,0.138498,0.698731
27,(Class_Crew),"(Age_Adult, Gender_Male)",0.402090,0.757383,0.391640,0.974011,1.286022,1.0,0.087104,9.335480,0.371976,0.510059,0.892882,0.745554
26,"(Age_Adult, Gender_Male)",(Class_Crew),0.757383,0.402090,0.391640,0.517097,1.286022,1.0,0.087104,1.238157,0.916706,0.510059,0.192348,0.745554
34,(Gender_Male),"(Class_Crew, Survived_No)",0.786461,0.305770,0.304407,0.387060,1.265851,1.0,0.063931,1.132622,0.983509,0.386390,0.117093,0.691301
31,"(Class_Crew, Survived_No)",(Gender_Male),0.305770,0.786461,0.304407,0.995542,1.265851,1.0,0.063931,47.903983,0.302519,0.386390,0.979125,0.691301
55,"(Class_Crew, Age_Adult, Survived_No)",(Gender_Male),0.305770,0.786461,0.304407,0.995542,1.265851,1.0,0.063931,47.903983,0.302519,0.386390,0.979125,0.691301
66,(Gender_Male),"(Class_Crew, Age_Adult, Survived_No)",0.786461,0.305770,0.304407,0.387060,1.265851,1.0,0.063931,1.132622,0.983509,0.386390,0.117093,0.691301


#####  An leverage value of 0 indicates independence. Range will be [-1 1]
A high conviction value means that the consequent is highly depending on the antecedent and range [0 inf]

In [13]:
rules.sort_values('lift',ascending = False)[0:20]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
8,(Gender_Female),(Survived_Yes),0.213539,0.323035,0.156293,0.731915,2.265745,1.0,0.087312,2.525187,0.710327,0.410992,0.603990,0.607870
9,(Survived_Yes),(Gender_Female),0.323035,0.213539,0.156293,0.483826,2.265745,1.0,0.087312,1.523634,0.825219,0.410992,0.343674,0.607870
60,"(Class_Crew, Survived_No)","(Age_Adult, Gender_Male)",0.305770,0.757383,0.304407,0.995542,1.314450,1.0,0.072822,54.427079,0.344592,0.401198,0.981627,0.698731
61,"(Age_Adult, Gender_Male)","(Class_Crew, Survived_No)",0.757383,0.305770,0.304407,0.401920,1.314450,1.0,0.072822,1.160764,0.986022,0.401198,0.138498,0.698731
27,(Class_Crew),"(Age_Adult, Gender_Male)",0.402090,0.757383,0.391640,0.974011,1.286022,1.0,0.087104,9.335480,0.371976,0.510059,0.892882,0.745554
26,"(Age_Adult, Gender_Male)",(Class_Crew),0.757383,0.402090,0.391640,0.517097,1.286022,1.0,0.087104,1.238157,0.916706,0.510059,0.192348,0.745554
34,(Gender_Male),"(Class_Crew, Survived_No)",0.786461,0.305770,0.304407,0.387060,1.265851,1.0,0.063931,1.132622,0.983509,0.386390,0.117093,0.691301
31,"(Class_Crew, Survived_No)",(Gender_Male),0.305770,0.786461,0.304407,0.995542,1.265851,1.0,0.063931,47.903983,0.302519,0.386390,0.979125,0.691301
55,"(Class_Crew, Age_Adult, Survived_No)",(Gender_Male),0.305770,0.786461,0.304407,0.995542,1.265851,1.0,0.063931,47.903983,0.302519,0.386390,0.979125,0.691301
66,(Gender_Male),"(Class_Crew, Age_Adult, Survived_No)",0.786461,0.305770,0.304407,0.387060,1.265851,1.0,0.063931,1.132622,0.983509,0.386390,0.117093,0.691301


In [14]:
rules[rules.lift>1]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Class_3rd),(Survived_No),0.320763,0.676965,0.239891,0.747875,1.104747,1.0,0.022745,1.281251,0.139592,0.316547,0.219513,0.551119
1,(Survived_No),(Class_3rd),0.676965,0.320763,0.239891,0.354362,1.104747,1.0,0.022745,1.052040,0.293515,0.316547,0.049466,0.551119
2,(Class_Crew),(Gender_Male),0.402090,0.786461,0.391640,0.974011,1.238474,1.0,0.075412,8.216621,0.322047,0.491448,0.878295,0.735995
3,(Gender_Male),(Class_Crew),0.786461,0.402090,0.391640,0.497978,1.238474,1.0,0.075412,1.191004,0.901730,0.491448,0.160372,0.735995
4,(Class_Crew),(Age_Adult),0.402090,0.950477,0.402090,1.000000,1.052103,1.0,0.019913,inf,0.082827,0.423040,1.000000,0.711520
5,(Age_Adult),(Class_Crew),0.950477,0.402090,0.402090,0.423040,1.052103,1.0,0.019913,1.036311,1.000000,0.423040,0.035039,0.711520
6,(Class_Crew),(Survived_No),0.402090,0.676965,0.305770,0.760452,1.123325,1.0,0.033569,1.348519,0.183616,0.395417,0.258446,0.606065
7,(Survived_No),(Class_Crew),0.676965,0.402090,0.305770,0.451678,1.123325,1.0,0.033569,1.090436,0.339858,0.395417,0.082935,0.606065
8,(Gender_Female),(Survived_Yes),0.213539,0.323035,0.156293,0.731915,2.265745,1.0,0.087312,2.525187,0.710327,0.410992,0.603990,0.607870
9,(Survived_Yes),(Gender_Female),0.323035,0.213539,0.156293,0.483826,2.265745,1.0,0.087312,1.523634,0.825219,0.410992,0.343674,0.607870


In [ ]:
'''Final Summary of Steps
Step	Description
Step 1	Collect transaction data
Step 2	Preprocess data (convert lists to one-hot encoding)
Step 3	Use the Apriori algorithm to find frequent itemsets (Support ≥ threshold)
Step 4	Generate association rules (Confidence & Lift)
Step 5	Interpret results for business applications
